## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
#include<bits/stdc++.h>
using namespace std;
const int N=1030;
int n,a,b,len;
int p[N],posi[N];
vector<pair<int,int> > ans;
int lowbit(int x)
{
    return x&-x;
}
void update_pos()
{
    for(int i=0;i<n;i++) posi[p[i]]=i;
}
void add_all(int x)
{
    x%=n;
    if(x<0) x+=n;
    if(x==0) return;

    ans.push_back({2,x});

    for(int i=0;i<n;i++) p[i]=(p[i]+x)%n;
    update_pos();
}
void xor_all(int x)
{
    if(x==0) return;

    ans.push_back({1,x});

    for(int i=0;i<n;i++) p[i]^=x;
    update_pos();
}
void swap_ab()
{
    ans.push_back({0,0});

    for(int i=0;i<n;i++)
    {
        if(p[i]==a) p[i]=b;
        else if(p[i]==b) p[i]=a;
    }

    update_pos();
}
// 找一组等价的位置，方便把要交换的两个数搬到 a,b 上
void get_place(int x,int y,int &u,int &v)
{
    int d=(y-x+n-len+n)%n;

    u=0;
    v=0;

    for(int step=n/2;step>=2*len;step>>=1)
    {
        if(d>=step)
        {
            d-=step;
            v+=step/2;
        }
        else
        {
            u+=step/2;
        }
    }

    u+=n/2;
    u+=(x&(len-1));
    v+=(x&(len-1));
}
void swap_value(int x,int y)
{
    if(x==y) return;

    if((x/len)%2==(y/len)%2)
    {
        // 如果两个数在同一边，就找一个同余数的中间数绕一下
        int mid;

        if((x/len)%2==0) mid=(x&(len-1))+len;
        else mid=(x&(len-1));

        swap_value(x,mid);
        swap_value(y,mid);
        swap_value(x,mid);

        return;
    }

    int pa,pb,px,py;

    get_place(a,b,pa,pb);
    get_place(x,y,px,py);

    // 先把 x,y 搬成 a,b
    add_all(px-x);
    xor_all(px^pa);
    add_all(a-pa);

    swap_ab();

    // 再把整体操作倒回去
    add_all(pa-a);
    xor_all(px^pa);
    add_all(x-px);
}
struct SmallPerm
{
    vector<int> val;
    vector<int> op; // 正数表示加法，负数表示异或

    bool build()
    {
        int m=val.size();

        vector<int> vis(m,0);

        for(int x:val)
        {
            if(x<0||x>=m||vis[x]) return false;
            vis[x]=1;
        }

        if(m==1) return true;

        int half=m/2;

        SmallPerm L,R;
        L.val.resize(half);
        R.val.resize(half);

        for(int i=0;i<half;i++)
        {
            L.val[i]=val[i*2]/2;
            R.val[i]=val[i*2+1]/2;
        }

        if(!L.build()) return false;
        if(!R.build()) return false;

        if(val[0]&1)
        {
            if(m==2) op.push_back(1);
            else op.push_back(-1);
        }

        int lx=0;

        for(int x:L.op)
        {
            if(x>0)
            {
                op.push_back(-1);
                op.push_back(1);
            }
            else
            {
                op.push_back(x*2);
                lx^=(-x)*2;
            }
        }

        if(lx) op.push_back(-lx);

        int rx=0;

        for(int x:R.op)
        {
            if(x>0)
            {
                op.push_back(1);
                op.push_back(-1);
            }
            else
            {
                op.push_back(x*2);
                rx^=(-x)*2;
            }
        }

        if((lx&half)!=(rx&half)) return false;

        if(lx>=half) lx-=half;
        if(rx>=half) rx-=half;

        if(lx!=rx) return false;

        // 连续两个异或可以合并，少输出几步
        vector<int> tmp;

        for(int x:op)
        {
            if(!tmp.empty()&&x<0&&tmp.back()<0)
            {
                int now=(-tmp.back())^(-x);
                tmp.pop_back();

                if(now) tmp.push_back(-now);
            }
            else
            {
                tmp.push_back(x);
            }
        }

        op.swap(tmp);

        return true;
    }
};
int main()
{
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    cin>>n;
    cin>>a>>b;

    for(int i=0;i<n;i++) cin>>p[i];

    update_pos();

    len=(a-b+n)%n;
    len=lowbit(len);

    if(len==0) len=n;

    // 先只看低位，把每个位置的余数调成正确的
    if(len>1)
    {
        SmallPerm base;
        base.val.resize(len);

        for(int i=0;i<len;i++) base.val[i]=p[i]&(len-1);

        if(!base.build())
        {
            cout<<-1<<"\n";
            return 0;
        }

        for(int x:base.op)
        {
            if(x>0) add_all(x);
            else xor_all(-x);
        }
    }

    // 余数类之间不能乱换了，所以每个余数类单独检查和排序
    for(int r=0;r<len;r++)
    {
        vector<int> v;

        for(int i=r;i<n;i+=len) v.push_back(p[i]);

        sort(v.begin(),v.end());

        int id=0;

        for(int i=r;i<n;i+=len)
        {
            if(v[id]!=i)
            {
                cout<<-1<<"\n";
                return 0;
            }

            id++;
        }

        for(int i=r;i<n;i+=len)
        {
            if(p[i]!=i) swap_value(i,p[i]);
        }
    }

    for(int i=0;i<n;i++)
    {
        if(p[i]!=i)
        {
            cout<<-1<<"\n";
            return 0;
        }
    }

    if((int)ans.size()>32768)
    {
        cout<<-1<<"\n";
        return 0;
    }

    cout<<ans.size()<<"\n";

    for(auto x:ans)
    {
        if(x.first==0) cout<<0<<"\n";
        else cout<<x.first<<" "<<x.second<<"\n";
    }

    return 0;
}

## B 长跑

In [ ]:
#include <stdio.h>
int main()
{
    int N,L,Maxn,S;
    while (scanf("%d %d %d %d", &N, &L, &Maxn, &S) == 4)
    {
        int p=Maxn,m=S,flag=0;
        int Pi[N],Ci[N];
        int rec=0;
        for(int i=0;i<N;i++)
        {
            scanf("%d %d",&Pi[i],&Ci[i]);
        }
        int temp=3001,min=L;
        for(int i=0;i<=L;i++)
        {
            int k=0;
            if(p>=L)
            {
                printf("Yes\n");
                flag=1;
                break;
            }
            if(i>p)
            {
                break;
            }
            for(int j=0;j<N;j++)
            {
                int gap=p-i;
                if(i==Pi[j]&&m>=Ci[j]&&gap<=min)
                {
                    if(gap==min&&temp>=Ci[j])
                    {
                        temp=Ci[j];
                        k=1;
                        min=p-i;
                        rec=i;
//                         printf("%d ",min);
                    }
                    else if(gap!=min)
                    {
                        temp=Ci[j];
                        k=1;
                        min=p-i;
                        rec=i;
//                         printf("%d ",min);
                    }
                }
            }
            if(k==1&&min>0)
            {    
//                printf("%d ",i);
               continue;
            }
            if(k==1&&min==0)
            {
                
                m-=temp;
                p=rec+Maxn;
                temp=3001;
                min=L;
            }
            if(k==0&&i==p)
            {
                m-=temp;
                p=rec+Maxn;
                temp=3001;
                min=L;
            }
        }
        if(flag==0)
        {
            printf("No\n");
        }
    }
    return 0;
}


## C 最长回文

In [ ]:
#include <bits/stdc++.h>
using namespace std;
vector<int> manacher(const string &s)
{
    int n = s.size();
    //处理@#a#b#c#$
    string t;
    t.reserve(2 * n + 3);
    t.push_back('@');
    for (char c : s) {
        t.push_back('#');
        t.push_back(c);
    }
    t.push_back('#');
    t.push_back('$');
    int m=t.size();
    vector<int> p(m,1);
    int center=0, right=0;
    for(int i=1;i<m-1;i++)
    {
        if(i<right)
        {
            p[i]=min(p[2*center-i],right-i);
        }
        while(t[i-p[i]]==t[i+p[i]])
        {
            p[i]++;
        }
        if(i+p[i]>right)
        {
            right=i+p[i];
            center=i;
        }
    }
    return p;
}
string build(const string &s)
{
    string t;
    t.reserve(2*s.size()+3);
    t.push_back('@');
    for (char c:s)
    {
        t.push_back('#');
        t.push_back(c);
    }
    t.push_back('#');
    t.push_back('$');
    return t;
}
int main()
{
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    int n;
    string A, B;
    cin >> n >> A >> B;
    string TA = build(A);
    string TB = build(B);
    vector<int> PA = manacher(A);
    vector<int> PB = manacher(B);
    int ans = 1;
    /*
        i是A中的回文中心
        i-2是和它对应的B中回文中心
        tmp表示当前处理后字符串里的回文半径
        最终真实长度=tmp-1
    */
    for(int i=2; i<=2*n+1;i++)
    {
        int tmp=max(PA[i],PB[i-2]);
        while (TA[i-tmp]==TB[i+tmp-2])
        {
            tmp++;
        }
        ans = max(ans, tmp);
    }
    cout<<ans-1<<'\n';
    return 0;
}

## D 优惠券

In [ ]:
#include <bits/stdc++.h>
using namespace std;
const int MAXX = 100000 + 5;
int lastPos[MAXX];
char lastOp[MAXX];
int main()
{
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    int m;
    while(cin>>m)
    {
        set<int> qpos;
        vector<int> touched;
        int ans = -1;
        for (int i = 1; i <= m; i++)
        {
            string op;
            cin>>op;
            if (op=="?"||op=="？")
            {
                if (ans==-1)
                {
                    qpos.insert(i);
                }
                continue;
            }
            int x;
            cin >> x;
            if (ans != -1)
            {
                continue;
            }
            char c = op[0];
            if (lastPos[x] == 0)
            {
                touched.push_back(x);
                if (c == 'O')
                {
                    if (qpos.empty())
                    {
                        ans=i;
                    } 
                    else
                    {
                        qpos.erase(qpos.begin());
                    }
                }
            }
            else
            {
                if (lastOp[x] == c)
                {
                    auto it=qpos.upper_bound(lastPos[x]);
                    if(it==qpos.end())
                    {
                        ans=i;
                    }
                    else
                    {
                        qpos.erase(it);
                    }
                }
            }
            if (ans==-1)
            {
                lastPos[x]=i;
                lastOp[x]=c;
            }
        }
        cout<<ans<<'\n';
        for (int x:touched)
        {
            lastPos[x]=0;
            lastOp[x]=0;
        }
    }
    return 0;
}

## E 任意点

In [ ]:
#include<iostream>
#include <vector>
using namespace std;
struct point
{
    int x;
    int y;
};
struct BCB
{
    //初始化父节点，每个坐标都是父节点
    vector <int> fa;
    BCB(int n)
    {
        fa.resize(n);
        for(int i=0;i<n;i++)
            fa[i]=i;
    }
    int find(int x)
    {
        //如果相同，就代表本身为该连通块的父节点
        if(fa[x]==x)
            return x;
        //更新联通后子节点的父亲节点
        return fa[x]=find(fa[x]);
    }
    void unions(int a,int b)
    {
        //a的父节点与b的父节点不同时，把a的父节点改成b
        if(find(a)!=find(b))
            fa[find(a)]=find(b);
    }
};
int main(void)
{
    int n;
    cin>>n;
    vector <point> a(n);
    for(int i=0;i<n;i++)
        cin>>a[i].x>>a[i].y;
    BCB B(n);
    //两两坐标互相联合，看是否合并
    for(int i=0;i<n;i++)
    {
        for(int j=i+1;j<n;j++)
        {
            if(a[i].x==a[j].x||a[i].y==a[j].y)
                B.unions(i,j);
        }
    }
    //计算联通块
    int block=0;
    for(int i=0;i<n;i++)
    {
        if(B.find(i)==i)
            block++;
    }
    cout<<block-1;
    return 0;
}

## F 通配符匹配

In [ ]:
#include <bits/stdc++.h>
using namespace std;

struct Block {
    string str;
    int offset;
    vector<int> pi;
};

struct Segment {
    string pat;
    int len;
    vector<Block> blocks;
};

vector<int> buildPi(const string& t) {
    vector<int> pi(t.size(), 0);
    for (int i = 1, j = 0; i < (int)t.size(); i++) {
        while (j > 0 && t[i] != t[j]) j = pi[j - 1];
        if (t[i] == t[j]) j++;
        pi[i] = j;
    }
    return pi;
}

Segment makeSegment(const string& x) {
    Segment seg;
    seg.pat = x;
    seg.len = x.size();

    for (int i = 0; i < seg.len; ) {
        if (x[i] == '?') {
            i++;
            continue;
        }

        int st = i;
        while (i < seg.len && x[i] != '?') i++;

        Block b;
        b.str = x.substr(st, i - st);
        b.offset = st;
        b.pi = buildPi(b.str);
        seg.blocks.push_back(b);
    }

    return seg;
}

bool matchAt(const Segment& seg, const string& s, int pos) {
    if (pos < 0) return false;
    if (pos + seg.len > (int)s.size()) return false;

    for (int i = 0; i < seg.len; i++) {
        if (seg.pat[i] != '?' && seg.pat[i] != s[pos + i]) {
            return false;
        }
    }

    return true;
}

vector<unsigned char> kmpOccur(const string& s, const Block& b) {
    int n = s.size();
    int m = b.str.size();

    vector<unsigned char> occ(n + 1, 0);

    if (m == 0 || m > n) return occ;

    for (int i = 0, j = 0; i < n; i++) {
        while (j > 0 && s[i] != b.str[j]) {
            j = b.pi[j - 1];
        }

        if (s[i] == b.str[j]) {
            j++;
        }

        if (j == m) {
            occ[i - m + 1] = 1;
            j = b.pi[j - 1];
        }
    }

    return occ;
}

// 在 s 的 [l, r) 区间中，找 seg 最早能匹配的位置
int findSegment(const Segment& seg, const string& s, int l, int r) {
    if (seg.len == 0) return l;
    if (r - l < seg.len) return -1;

    // 整段全是 '?'
    if (seg.blocks.empty()) return l;

    vector<vector<unsigned char>> occs;
    occs.reserve(seg.blocks.size());

    for (const auto& b : seg.blocks) {
        occs.push_back(kmpOccur(s, b));
    }

    int maxStart = r - seg.len;

    for (int pos = l; pos <= maxStart; pos++) {
        bool ok = true;

        for (int i = 0; i < (int)seg.blocks.size(); i++) {
            int needPos = pos + seg.blocks[i].offset;
            if (!occs[i][needPos]) {
                ok = false;
                break;
            }
        }

        if (ok) return pos;
    }

    return -1;
}

bool isMatch(const string& p, const string& s) {
    vector<string> raw;
    string cur;
    bool hasStar = false;

    for (char c : p) {
        if (c == '*') {
            hasStar = true;
            raw.push_back(cur);
            cur.clear();
        } else {
            cur.push_back(c);
        }
    }
    raw.push_back(cur);

    vector<Segment> segs;
    for (const string& x : raw) {
        segs.push_back(makeSegment(x));
    }

    // 没有 '*'
    if (!hasStar) {
        if ((int)p.size() != (int)s.size()) return false;
        return matchAt(segs[0], s, 0);
    }

    int n = s.size();

    // 前缀段
    if (!matchAt(segs.front(), s, 0)) return false;
    int left = segs.front().len;

    // 后缀段
    int suffixLen = segs.back().len;
    int suffixPos = n - suffixLen;
    if (!matchAt(segs.back(), s, suffixPos)) return false;
    int right = suffixPos;

    if (left > right) return false;

    // 中间段依次找
    for (int i = 1; i + 1 < (int)segs.size(); i++) {
        int pos = findSegment(segs[i], s, left, right);
        if (pos == -1) return false;
        left = pos + segs[i].len;
    }

    return left <= right;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    string pattern;
    cin >> pattern;

    int n;
    cin >> n;

    while (n--) {
        string filename;
        cin >> filename;

        cout << (isMatch(pattern, filename) ? "YES" : "NO") << '\n';
    }

    return 0;
}

## G 汉诺塔

In [ ]:
#include<iostream>
#include<string>
#include<math.h>
using namespace std;
int main(void)
{
    int n;
    cin>>n;
    string op[6];
    long long f[n+1][3];
    long long to[n+1][3];//目标落在哪个盘上
    for(int i=0;i<6;i++)
        cin>>op[i];
    //初始化一个盘子
    for(int s=0;s<3;s++)
    {
        for(int i=0;i<6;i++)
        {
            int start=op[i][0]-'A';
            if(start==s)
            {
                f[1][s]=1;
                int end=op[i][1]-'A';
                to[1][s]=end;
                break;
            }         
        }
    }
    //开始递归
    for(int i=2;i<=n;i++)
    {
        for(int s=0;s<3;s++)
        {
            int p1=to[i-1][s];//i-1个盘子移动到柱子p1
            int p2=3-p1-s;//最大盘子i移动到剩余柱子p2
            int end=to[i-1][p1];//i-1个盘子移动到下一步盘子
            //假如i-1个盘子的优先级刚好能从p1移动到p2的最大盘子上
            if(end==p2)
            {
                f[i][s]=f[i-1][s]+1+f[i-1][p1];
                to[i][s]=p2;
            }
            //end需要先移动S的柱子上，把最大盘子i从p2移动到p1,最终把i-1个盘子从s移动到p1上
            else
            {
                f[i][s]=f[i-1][s]+1+f[i-1][p1]+1+f[i-1][s];
                to[i][s]=p1;
            }
        }
    }
    cout<<f[n][0];
    return 0;
}

## H 马步距离

In [ ]:
#include<iostream>
#include<cmath>
using namespace std;
int main(void)
{
    int xp,yp,xs,ys;
    cin>>xp>>yp>>xs>>ys;
    int x=abs(xp-xs);
    int y=abs(yp-ys);
    int k;//定义步数
    //限制在第一象限的八分之一区域
    if(x<y)
        swap(x,y);
    //两个特殊情况
    if(x==1&&y==0)
    {
        cout<<3;
        return 0;
    }
        
    if(x==2&&y==2)
    {
        cout<<4;
        return 0;
    }
    k=max((x+1)/2,(x+y+2)/3);
    if((x+y)%2==0&&k%2==1)
        k++;
    else if((x+y)%2==1&&k%2==0)
        k++;
    cout<<k;
    return 0;
}

## I 直方图最大矩形

In [ ]:
class Solution {
public:
    /**
     * 代码中的类名、方法名、参数名已经指定，请勿修改，直接返回方法规定的值即可
     *
     * 
     * @param heights int整型vector 
     * @return int整型
     */
    int largestRectangleArea(vector<int>& heights) {
        // write code here
        int max=0;
        stack<int> a;
        int left=0,right=0,height=0,width=0,area=0;
        a.push(0);
        heights.push_back(0);//末尾0停止标志
        for(int i=1;i<heights.size();i++)
        {
            //没找到右边界就一直入栈
            if(heights[a.top()]<=heights[i])
            {
                a.push(i);
            }
            //找到右边界
            else
            {
                //记录右边界下标与当前栈顶元素高度
                while(1)
                {
                    if(a.size()==0)
                        break;
                    //第二次循环后当前元素符合入栈条件
                    if(heights[a.top()]<=heights[i])
                    {
                        a.push(i);
                        break;
                    }
                    height=heights[a.top()];
                    right=i;
                    a.pop();
                    //找到比height更小的左边界
                    if(a.size()!=0&&heights[a.top()]<height)
                    {
                        left=a.top();
                        width=right-left-1;
                        area=width*height;
                        if(area>max)
                            max=area;
                    }
                    //栈空的情况
                    else if(a.size()==0)
                    {
                        left=-1;
                        width=right-left-1;
                        area=width*height;
                        if(area>max)
                            max=area;
                    }
                }
                a.push(i);
            }
        }
        return max;
    }
};

## J 消防局的设立

In [ ]:
#include<iostream>
#include<vector>
#include<map>
#include<algorithm>
using namespace std;
void cover(int x,vector<int> &iscover,map<int,vector<int>> &child,vector<int> &fa)
{
    //祖父节点本身被覆盖:x
    iscover[x]=1;
    //祖父节点的子节点被覆盖：x-1
    for(int v:child[x])
        iscover[v]=1;
    //祖父节点的孙子节点被覆盖：x-2
    for(int i:child[x])
        for(int v:child[i])
            iscover[v]=1;
    //祖父节点的父节点被覆盖：x+1
    iscover[fa[x]]=1;
    //祖父节点的祖父节点被覆盖：x+2
    iscover[fa[fa[x]]]=1;
    //祖父节点的兄弟节点被覆盖：x+2
    for(int v:child[fa[x]])
        iscover[v]=1;
}
int main(void)
{
    int n;
    cin>>n;
    vector <int> a(n+1,0);
    vector <int> fa(n+1,0);//记录i的父节点
    map <int,vector<int>> child;//记录i的孩子节点
    vector <int> dep(n+1,0);//记录i的深度
    vector<int> node(n+1);//记录节点，方便后续深度排序
    for(int i=0;i<=n;i++)
        node[i]=i;
    vector <int> iscover(n+1,0);//记录i是否被覆盖
    int num=0;//消防局数量
    for(int i=2;i<=n;i++)
        cin>>a[i];
    //题目设置a[i]<i
    for(int i=2;i<=n;i++)
    {
        fa[i]=a[i];//将i的父节点指向a[i]
        child[fa[i]].push_back(i);//将fa[i]的子节点指向i
        dep[i]=dep[fa[i]]+1;//i的深度是父节点深度+1
    }
    //对深度进行排序
    sort(node.begin()+1,node.end(),[&dep](int a,int b)
         {
             return dep[a]>dep[b];
         });
    //从深度>2的开始处理
    for(int i=1;i<=n;i++)
    {
        if(dep[node[i]]>2&&iscover[node[i]]==0)
        {
            cover(fa[fa[node[i]]],iscover,child,fa);//将i的祖父节点建造消防局
            num++;
        }
    }
    //检查是否所有节点被覆盖
    bool flag=false;
    for(int i=1;i<=n;i++)
    {
        if(iscover[i]!=1)
            flag=true;
    }
    if(flag)
        num++;
    cout<<num;
    return 0;
}